# DPO (Direct Preference Optimization) 算法详解与代码实现

在大型语言模型 (LLM) 的人类偏好对齐(Alignment)过程中，传统的手段是 **RLHF (Reinforcement Learning from Human Feedback)**：即先利用人类标注数据训练一个独立的**奖励模型 (Reward Model)**，然后利用 PPO 这样的强化学习算法，不断让模型去最大化这个奖励模型给出的分数。这一过程不仅引入了多个模型网络（ Actor, Critic, Reward Model, Reference Model等），极其占用显存，而且因为 PPO 对超参数极其敏感，导致训练极不稳定。

**DPO (Direct Preference Optimization，直接偏好优化)** 在提出后引起了学术界与工业界的轰动。它巧妙地通过数学推导（依据 Bradley-Terry 偏好模型），将 RLHF 中的 RL(强化学习) 和 RM(奖励模型) 阶段**合二为一**。直接在语言模型原本的监督微调（SFT）基础上，把人类给出的**偏好数据对 $(x, y_w, y_l)$** (也就是好的回答 $y_w$ 和坏的回答 $y_l$ ) 作为输入进行训练。

---

## 1. DPO 的理论推导核心

传统的 RLHF 中，奖励模型 $r_{\phi}(x, y)$ 是这样拟合偏好关系的（Bradley-Terry模型）：
$$ p(y_w \succ y_l | x) = \sigma (r(x, y_w) - r(x, y_l)) $$

在此框架下，PPO 试图找到一个策略 $\pi_{\theta}$ 去最大化奖励：
$$ \max_{\pi_\theta} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi_\theta} [r(x, y)] - \beta \mathbb{D}_{KL} [\pi_\theta (y|x) || \pi_{\text{ref}}(y|x)] $$

**DPO 的神来之笔**：作者发现，如果我们在上面这个带 KL 惩罚的最大化公式中求全局解析解（最优分配），最优的生成概率 $\pi^*(y|x)$ 和奖励函数 $r(x,y)$ 存在以下精确的解析映射关系：
$$ r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x) $$

我们直接将这个 $r(x,y)$ 的等价物代回到 Bradley-Terry 偏好模型里（ $\sigma(r_w - r_l)$ ），配分函数 $Z(x)$ 因为两者相减而完美抵消了！因此得到了 DPO 的最终极大似然损失目标函数：

$$ \mathcal{L}_{DPO}(\pi_\theta; \pi_{\text{ref}}) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right] $$

### DPO 到底在干什么？
- $\beta$ : 就是控制偏离参考模型的惩罚拉力系数（常见设定如 0.1）。
- $\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}$ : 模型对于**好回答($y_w$)** 的隐含奖励打分。
- $\log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}$ : 模型对于**差回答($y_l$)** 的隐含奖励打分。
- DPO 的本质其实就是：**增大“当前网络生成好回答的概率 相较于 参考模型”的倍率；同时打压“生成坏回答概率”的倍率。**并用一个 Sigmoid ($\sigma$) 函数包装成二分类交叉熵。

---

## 2. DPO 的 PyTorch 核心代码实现

为了直观体会 DPO，我们将用一套伪 NLP 环境来实现它的底层逻辑。我们随机生成一批 `query` 以及对应的 `win_response` 和 `lose_response`。在真实的 LLM 开源库（比如 HuggingFace TRL）中，这套逻辑其实就是 `DPOTrainer` 内部底层的运行算法。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleLanguageModel(nn.Module):
    """
    为了演示流程而建立的一个极其精简的因果语言模型。
    输入 input_ids, 经过一层 Embedding 和一个隐层线性变换预测 Next Token。
    真实应用中，这通常是一个几十个Transformer Block组成的庞然大物 (如 Llama 3)。
    """
    def __init__(self, vocab_size, embed_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # 用单层替代 Transformer 抽取序列特征
        self.lstm = nn.LSTM(embed_size, embed_size, batch_first=True)
        # 语言模型的 LM Head：把隐藏状态投射为词表上每个单词的概率分布
        self.lm_head = nn.Linear(embed_size, vocab_size)
        
    def forward(self, input_ids):
        # input_ids: (batch_size, sequence_length)
        x = self.embedding(input_ids)
        output, _ = self.lstm(x)    # output: (batch, seq_len, embed_size)
        logits = self.lm_head(output) # logits: (batch, seq_len, vocab_size)
        return logits

def get_batch_logps(logits, labels, label_pad_token_id=-100):
    """
    计算基于当前概率分布(logits)产生目标文本(labels)的正对数似然 (Log-Probability)
    
    参数:
        logits: 模型输出未归一化的特征张量, 形状 (batch, seq_len, vocab_size)
        labels: 对应希望模型输出的正确标签, 即 ground truth ID 序列。
                对模型不想学习或属于提示词(Prompt)的部分，通常填 label_pad_token_id 屏蔽。
    """
    # 丢掉最后一个 logit（因为它只能用来猜测越界的再下一个 token）
    # 同时忽略 labels 的最前面一个 token (它的概率来自于更前面的文段或者没得学)
    logits = logits[:, :-1, :]
    labels = labels[:, 1:].clone()
    
    # 构建一个掩码，跳过那些我们不需要计算梯度的位置（如 Padding，或是前缀的 Query 部分）
    loss_mask = labels != label_pad_token_id
    
    # 巧妙转换：因为交叉熵里需要被计算目标的ID不能是 -100，不然gather/cross_entropy内部会强行越界，因此设个 dummy id (0) 
    labels[labels == label_pad_token_id] = 0

    # 对 logits 算 log_softmax, 即将所有词的打分转化为符合归一化的 log(P) 概率值
    per_token_logps = F.log_softmax(logits, dim=-1)
    
    # per_token_logps 有多大的候选词表，使用 gather，用真实存在的 label 索引，选出“正确答案”对应的对数概率！
    # 并在最后 squeeze 摊平张量维度。
    per_token_logps = torch.gather(per_token_logps, dim=2, index=labels.unsqueeze(2)).squeeze(2)

    # 剔除掉属于屏蔽 padding 或者 prompt的 logps（它们与 mask 乘上后直接为 0.0）
    # 并聚合这段序列内每句话的总预测概率：sum
    return (per_token_logps * loss_mask).sum(-1)


# ==========================================
# 核心 DPO Loss 计算模块
# ==========================================
def dpo_loss_function(policy_chosen_logps, policy_rejected_logps,
                      ref_chosen_logps, ref_rejected_logps,
                      beta):
    """
    计算前向的 DPO (Direct Preference Optimization) 损失。
    """
    # 1. 计算被惩罚(KL Penalty)保护后的偏好概率增量值
    # policy_chosen_logps - ref_chosen_logps 就是论文里所谓的"拟合出该样本的隐式奖励"打分
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps
    
    # 这个 diff 就代表我们的当前策略在好的样本上的自信度和差样本的自信度之差，
    # 是否足够领先参考模型(Reference)。领先的越少(或落后)，最后算出的惩罚也就越大。
    logits = pi_logratios - ref_logratios
    
    # 2. 乘以拉力系数 beta 并做 Sigmoid(或者用二元交叉熵-log sigmoid 等价等大)
    # L_DPO = - log( sigmoid(beta * (奖励之差)) )
    losses = -F.logsigmoid(beta * logits)
    
    # 3. 统计一下两者对于获胜方在暗中预测出来的隐式奖励。方便监控打印
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps).detach()
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps).detach()
    
    # 返回当批次的平均损失和奖励信息
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()


# ---------------------------------------------
# DPO 训练配置与初始化
# ---------------------------------------------
torch.manual_seed(42)

vocab_size = 1000
embed_size = 128
beta = 0.1   # DPO 中经典惩罚常数 default 为 0.1
lr = 1e-4

# 当前我们要训练的模型 (策略网络), 此前理应被 SFT 预训练过
policy_model = SimpleLanguageModel(vocab_size, embed_size)

import copy
# DPO 的另一核心要素：冻结的参考模型。
# 它直接复制于原本的 SFT 模型，且不开启梯度求导，全程用于稳定训练锚点
reference_model = copy.deepcopy(policy_model)
reference_model.eval() 

optimizer = torch.optim.Adam(policy_model.parameters(), lr=lr)

---
### 模拟 DPO 的微型循环机制

在标准训练流程中：
1. 我们给网络输入包含 `Query（提示词） + Response` 完整拼接在一起的数据序列。
2. 然而 DPO 是要在生成的语段(Response)上判定好与坏。这意味着计算序列似然时不能牵连到前排提示(Query)的数据，这就通过我们刚才书写的 `get_batch_logps` 中 `label_pad_token_id=-100` 来实现了对损失掩盖（Loss Masking）。

In [2]:
epochs = 50
batch_size = 64
seq_length = 20

# [伪数据构造部分]
# 在真实数据集中这分别是一对偏好对，如 
# chosen = "你好，我是助手"
# rejected = "你这蠢驴"
# 他们共同前接同一个前缀查询 "你叫什么名字"
def get_dummy_batch():
    # 随机生成虚假的 chosen 和 rejected 数据序列
    chosen_ids = torch.randint(0, vocab_size, (batch_size, seq_length))
    rejected_ids = torch.randint(0, vocab_size, (batch_size, seq_length))
    
    # 设定 dummy_labels，我们将前 5 个 token 视作 Query (屏蔽学习), 所以将前继的 5 个位目标记为 -100
    chosen_labels = chosen_ids.clone()
    chosen_labels[:, :5] = -100 
    
    rejected_labels = rejected_ids.clone()
    rejected_labels[:, :5] = -100
    
    return chosen_ids, rejected_ids, chosen_labels, rejected_labels

print("=== 开始 DPO 简易演示训练 ===")
for step in range(epochs):
    chosen_ids, rejected_ids, chosen_labels, rejected_labels = get_batch_logps_dummy_data = get_dummy_batch()
    
    # ----------------------------------------------------
    # 步骤 1：让不更新梯度的 参考模型 分别测算 两种数据的先验打分能力作为锚点
    # ----------------------------------------------------
    with torch.no_grad():
        ref_chosen_logits = reference_model(chosen_ids)
        ref_rejected_logits = reference_model(rejected_ids)
        
        ref_chosen_logps = get_batch_logps(ref_chosen_logits, chosen_labels)
        ref_rejected_logps = get_batch_logps(ref_rejected_logits, rejected_labels)
        
    # ----------------------------------------------------
    # 步骤 2：对我们希望优化的策略网络 (Actor) 测算对数概率
    # ----------------------------------------------------
    policy_chosen_logits = policy_model(chosen_ids)
    policy_rejected_logits = policy_model(rejected_ids)
    
    policy_chosen_logps = get_batch_logps(policy_chosen_logits, chosen_labels)
    policy_rejected_logps = get_batch_logps(policy_rejected_logits, rejected_labels)
    
    # ----------------------------------------------------
    # 步骤 3：送入 DPO 万能交叉熵损失函数开始约束
    # ----------------------------------------------------
    loss, cho_reward, rej_reward = dpo_loss_function(
        policy_chosen_logps, policy_rejected_logps,
        ref_chosen_logps, ref_rejected_logps,
        beta=beta
    )
    
    # ----------------------------------------------------
    # 反向传播开始降梯
    # ----------------------------------------------------
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 10 == 0:
        # 当 loss 开始变小，而 reward_margin 开始拉大（chosen远大于rejected），便证明 DPO 吸取了人类偏好的经验正在生效
        print(f"Epoch {step+1}/{epochs} | DPO Loss: {loss.item():.4f} | "
              f"Implicit Reward (Chosen): {cho_reward:.4f} | Implicit Reward (Rejected): {rej_reward:.4f}")

print("=== DPO 训练结束 ===")

=== 开始 DPO 简易演示训练 ===
Epoch 10/50 | DPO Loss: 0.6931 | Implicit Reward (Chosen): 0.0000 | Implicit Reward (Rejected): -0.0000
Epoch 20/50 | DPO Loss: 0.6931 | Implicit Reward (Chosen): 0.0001 | Implicit Reward (Rejected): 0.0001
Epoch 30/50 | DPO Loss: 0.6931 | Implicit Reward (Chosen): 0.0002 | Implicit Reward (Rejected): 0.0000
Epoch 40/50 | DPO Loss: 0.6931 | Implicit Reward (Chosen): -0.0000 | Implicit Reward (Rejected): -0.0000
Epoch 50/50 | DPO Loss: 0.6932 | Implicit Reward (Chosen): 0.0001 | Implicit Reward (Rejected): 0.0003
=== DPO 训练结束 ===
